In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Lancer ce notebook depuis la racine du projet ou depuis src/.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python sources: {SRC_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from inference import load_diffusion_model
from purification.config import PurificationConfig
from purification.purifier import Purifier
from purification.step_curves import (
    linear_timesteps,
    quadratic_timesteps,
    uniform_log_snr_steps,
)
from purification.sweep import PurificationBatch, PurificationSweep
from utils import normalize_patch

## Modèle et données

`MAX_PATCHES` limite volontairement la démonstration. Mettre `None` pour charger tous les patchs de test. Aucune annotation n'est nécessaire pour exécuter la purification.

In [ ]:
DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
MODEL_PATH = "../models/unet32.pt"
TEST_DIR = PROJECT_ROOT / "dataset" / "test"
MAX_PATCHES = 32
BATCH_SIZE = 16
SEEDS = [0]

model, schedule = load_diffusion_model(MODEL_PATH, device=DEVICE)
print(f"Device: {DEVICE}")
print(f"Diffusion steps: {schedule.timesteps}")

In [ ]:
patch_paths = sorted(TEST_DIR.glob("*.npy"))
if MAX_PATCHES is not None:
    patch_paths = patch_paths[:MAX_PATCHES]
if not patch_paths:
    raise FileNotFoundError(f"Aucun patch trouvé dans {TEST_DIR}")

patches: dict[str, np.ndarray] = {}
for patch_path in patch_paths:
    patch = np.load(patch_path).astype(np.float32).squeeze()
    if patch.shape != (48, 48):
        raise ValueError(f"Forme invalide pour {patch_path.name}: {patch.shape}")
    patches[patch_path.name] = patch

print(f"{len(patches)} patchs chargés")

## Configurations de purification

Les courbes linéaire et uniforme en log-SNR respectent actuellement l'interface `StepCurve(K, t_start, schedule)`. `quadratic_timesteps` est importée pour rendre l'API visible, mais elle ne doit être ajoutée à `STEP_CURVES` qu'après alignement de sa signature sur cette interface.

In [ ]:
T_START_VALUES = [400, 600]
K_VALUES = [4, 8]
STEP_CURVES = [linear_timesteps, uniform_log_snr_steps, quadratic_timesteps]

configs = [
    PurificationConfig(t_start=t_start, K=K, step_curve=step_curve)
    for t_start in T_START_VALUES
    for K in K_VALUES
    for step_curve in STEP_CURVES
]
configs_by_name = {config.name: config for config in configs}

if len(configs_by_name) != len(configs):
    raise ValueError("Les noms de configuration ne sont pas uniques")

list(configs_by_name)

In [ ]:
for config in configs:
    steps = config.step_curve(config.K, config.t_start, schedule)
    assert len(steps) == config.K
    assert steps[0] == config.t_start
    assert all(current > following for current, following in zip(steps, steps[1:]))
    print(f"{config.name:28s} -> {steps}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for config in configs:
    steps = config.step_curve(config.K, config.t_start, schedule)
    ax.plot(range(len(steps)), steps, marker="o", label=config.name)
ax.set_xlabel("Étape DDIM")
ax.set_ylabel("Timestep de diffusion")
ax.set_title("Courbes de timesteps testées")
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## Utiliser une configuration sélectionnée

La sélection se fait par son nom stable. Le bruit est fourni explicitement afin de rendre la reconstruction reproductible et de pouvoir réutiliser exactement le même bruit avec une autre configuration.

In [ ]:
selected_name = configs[0].name
selected_config = configs_by_name[selected_name]
patch_name = sorted(patches)[0]

x0 = normalize_patch(torch.from_numpy(patches[patch_name]))[None, None].to(DEVICE)
generator = torch.Generator(device=DEVICE).manual_seed(SEEDS[0])
eps = torch.randn(x0.shape, generator=generator, dtype=x0.dtype, device=DEVICE)

purifier = Purifier(model=model, schedule=schedule)
x0_hat = purifier.reconstruct(x0=x0, config=selected_config, eps=eps)

print(f"Patch: {patch_name}")
print(f"Configuration: {selected_config.name}")
print(f"Input shape: {tuple(x0.shape)}")
print(f"Reconstruction shape: {tuple(x0_hat.shape)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(x0[0, 0].detach().cpu(), aspect="auto", cmap="viridis")
axes[0].set_title("Patch normalisé")
axes[1].imshow(x0_hat[0, 0].detach().cpu(), aspect="auto", cmap="viridis")
axes[1].set_title(selected_config.name)
for axis in axes:
    axis.set_axis_off()
plt.tight_layout()

## Comparer plusieurs configurations avec `PurificationSweep`

In [ ]:
sweep = PurificationSweep(
    model=model,
    schedule=schedule,
    configs=configs,
    patches=patches,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    seeds=SEEDS,
)

reconstructions: dict[tuple[str, str, int], torch.Tensor] = {}
normalized_inputs: dict[str, torch.Tensor] = {}

for batch in sweep.run():
    assert len(batch.patch_names) == batch.x0.size(0) == batch.x0_hat.size(0)
    for index, current_patch_name in enumerate(batch.patch_names):
        key = (current_patch_name, batch.config.name, batch.seed)
        reconstructions[key] = batch.x0_hat[index].detach().cpu()
        normalized_inputs.setdefault(
            current_patch_name, batch.x0[index].detach().cpu()
        )

expected = len(patches) * len(configs) * len(SEEDS)
assert len(reconstructions) == expected
print(f"{len(reconstructions)} reconstructions stockées sur CPU")

In [ ]:
# Comparer les reconstructions et leurs différences avec des échelles communes.
target_patch = sorted(patches)[25]
target_seed = SEEDS[0]

original = normalized_inputs[target_patch][0]
config_reconstructions = [
    reconstructions[(target_patch, config.name, target_seed)][0]
    for config in configs
]
signed_differences = [original - reconstruction for reconstruction in config_reconstructions]
absolute_differences = [difference.abs() for difference in signed_differences]
normalized_residuals = [
    absolute / (1 + reconstruction.abs())
    for absolute, reconstruction in zip(absolute_differences, config_reconstructions)
]

# Une échelle partagée par ligne évite l'autoscaling trompeur de imshow.
image_stack = torch.stack([original, *config_reconstructions])
image_vmin = torch.quantile(image_stack, 0.01).item()
image_vmax = torch.quantile(image_stack, 0.99).item()

signed_stack = torch.stack(signed_differences)
signed_limit = torch.quantile(signed_stack.abs(), 0.99).item()
absolute_vmax = torch.quantile(torch.stack(absolute_differences), 0.99).item()
residual_vmax = torch.quantile(torch.stack(normalized_residuals), 0.99).item()

n_columns = len(configs) + 1
fig, axes = plt.subplots(
    4,
    n_columns,
    figsize=(3 * n_columns, 11),
    constrained_layout=True,
)

image_plot = axes[0, 0].imshow(
    original,
    cmap="viridis",
    vmin=image_vmin,
    vmax=image_vmax,
    aspect="equal",
)
axes[0, 0].set_title("Original normalisé")
axes[1, 0].text(0.5, 0.5, "x0 - x0_hat", ha="center", va="center")
axes[2, 0].text(0.5, 0.5, "|x0 - x0_hat|", ha="center", va="center")
axes[3, 0].text(
    0.5,
    0.5,
    "|x0 - x0_hat| / (1 + |x0_hat|)",
    ha="center",
    va="center",
)

signed_plot = absolute_plot = residual_plot = None
for column, (config, reconstruction, signed, absolute, residual) in enumerate(
    zip(
        configs,
        config_reconstructions,
        signed_differences,
        absolute_differences,
        normalized_residuals,
    ),
    start=1,
):
    axes[0, column].imshow(
        reconstruction,
        cmap="viridis",
        vmin=image_vmin,
        vmax=image_vmax,
        aspect="equal",
    )
    axes[0, column].set_title(config.name)
    signed_plot = axes[1, column].imshow(
        signed,
        cmap="coolwarm",
        vmin=-signed_limit,
        vmax=signed_limit,
        aspect="equal",
    )
    absolute_plot = axes[2, column].imshow(
        absolute,
        cmap="magma",
        vmin=0,
        vmax=absolute_vmax,
        aspect="equal",
    )
    residual_plot = axes[3, column].imshow(
        residual,
        cmap="magma",
        vmin=0,
        vmax=residual_vmax,
        aspect="equal",
    )

for axis in axes.flat:
    axis.set_xticks([])
    axis.set_yticks([])

fig.colorbar(image_plot, ax=axes[0, :].tolist(), shrink=0.75, label="Amplitude")
fig.colorbar(signed_plot, ax=axes[1, 1:].tolist(), shrink=0.75, label="Différence signée")
fig.colorbar(absolute_plot, ax=axes[2, 1:].tolist(), shrink=0.75, label="Différence absolue")
fig.colorbar(residual_plot, ax=axes[3, 1:].tolist(), shrink=0.75, label="Résiduel normalisé")
fig.suptitle(f"{target_patch} — seed {target_seed}", fontsize=14)